In [1]:
import crystal_toolkit
import numpy as np
from multiprocessing.dummy import Pool
from pymatgen.ext.matproj import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core.surface import get_symmetrically_distinct_miller_indices
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

API_KEY = "XanMdfsODEDnGK6IKJhcmPGQ1sW13CsE"

with MPRester(API_KEY) as mpr:
    zno_bulk = mpr.get_structure_by_material_id("mp-2133")
    mgo_bulk = mpr.get_structure_by_material_id("mp-1265")

#Standardize the Structures
zno_bulk = SpacegroupAnalyzer(zno_bulk).get_conventional_standard_structure()
mgo_bulk = SpacegroupAnalyzer(mgo_bulk).get_conventional_standard_structure()

/home/akanksha/Siesta_5/miniconda3/envs/matsci-ct311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Enumerate Miller Planes - limiting Miller index to 2.

max_index = 1

zno_planes = get_symmetrically_distinct_miller_indices(zno_bulk, max_index)
mgo_planes = get_symmetrically_distinct_miller_indices(mgo_bulk, max_index)

print("ZnO planes:", zno_planes)
print("MgO planes:", mgo_planes)

ZnO planes: [(1, 1, 1), (1, 1, 0), (1, 0, 1), (1, 0, 0), (0, 0, 1)]
MgO planes: [(1, 1, 1), (1, 1, 0), (1, 0, 0)]


In [3]:
from pymatgen.analysis.interfaces.zsl import ZSLGenerator, ZSLMatch
from pymatgen.analysis.interfaces.substrate_analyzer import SubstrateAnalyzer, SubstrateMatch

zsl = ZSLGenerator(
    max_area_ratio_tol = 0.15,
    max_area = 200,
    max_length_tol = 0.05,
    max_angle_tol = 0.05
)

In [4]:
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

cib_objects = []

for mgo_plane in mgo_planes:
    for zno_plane in zno_planes:
        
        try:
            cib = CoherentInterfaceBuilder(
                substrate_structure=mgo_bulk,
                film_structure=zno_bulk,
                substrate_miller=mgo_plane,
                film_miller=zno_plane,
                zslgen=zsl
            )

            cib_objects.append({
                "zno_plane": zno_plane,
                "mgo_plane": mgo_plane,
                "cib": cib
            })

        except:
            continue

print("Total CIB objects:", len(cib_objects))

Total CIB objects: 15


In [5]:
terminations = cib.terminations
print("terminations:", len(terminations))
for t in terminations:
    print(t)

terminations: 2
('Zn_P6/mmm_1', 'MgO_P4/mmm_2')
('O2_P6/mmm_1', 'MgO_P4/mmm_2')


In [6]:
termination_info = []

for item in cib_objects:
    for termination in terminations:
    
        cib = item["cib"]
        n_terminations = len(cib.terminations)

        termination_info.append({
            "zno_plane": item["zno_plane"],
            "mgo_plane": item["mgo_plane"],
            "terminations": n_terminations
        })

termination_info

[{'zno_plane': (1, 1, 1), 'mgo_plane': (1, 1, 1), 'terminations': 1},
 {'zno_plane': (1, 1, 1), 'mgo_plane': (1, 1, 1), 'terminations': 1},
 {'zno_plane': (1, 1, 0), 'mgo_plane': (1, 1, 1), 'terminations': 1},
 {'zno_plane': (1, 1, 0), 'mgo_plane': (1, 1, 1), 'terminations': 1},
 {'zno_plane': (1, 0, 1), 'mgo_plane': (1, 1, 1), 'terminations': 2},
 {'zno_plane': (1, 0, 1), 'mgo_plane': (1, 1, 1), 'terminations': 2},
 {'zno_plane': (1, 0, 0), 'mgo_plane': (1, 1, 1), 'terminations': 1},
 {'zno_plane': (1, 0, 0), 'mgo_plane': (1, 1, 1), 'terminations': 1},
 {'zno_plane': (0, 0, 1), 'mgo_plane': (1, 1, 1), 'terminations': 2},
 {'zno_plane': (0, 0, 1), 'mgo_plane': (1, 1, 1), 'terminations': 2},
 {'zno_plane': (1, 1, 1), 'mgo_plane': (1, 1, 0), 'terminations': 1},
 {'zno_plane': (1, 1, 1), 'mgo_plane': (1, 1, 0), 'terminations': 1},
 {'zno_plane': (1, 1, 0), 'mgo_plane': (1, 1, 0), 'terminations': 1},
 {'zno_plane': (1, 1, 0), 'mgo_plane': (1, 1, 0), 'terminations': 1},
 {'zno_plane': (1, 0

In [7]:
match_info = []

for item in cib_objects:
    cib = item["cib"]
    matches = cib.zsl_matches
    match_info.append({
        "zno_plane": item["zno_plane"],
        "mgo_plane": item["mgo_plane"],
        "matches": len(matches)
    })

match_info

[{'zno_plane': (1, 1, 1), 'mgo_plane': (1, 1, 1), 'matches': 33},
 {'zno_plane': (1, 1, 0), 'mgo_plane': (1, 1, 1), 'matches': 114},
 {'zno_plane': (1, 0, 1), 'mgo_plane': (1, 1, 1), 'matches': 229},
 {'zno_plane': (1, 0, 0), 'mgo_plane': (1, 1, 1), 'matches': 143},
 {'zno_plane': (0, 0, 1), 'mgo_plane': (1, 1, 1), 'matches': 556},
 {'zno_plane': (1, 1, 1), 'mgo_plane': (1, 1, 0), 'matches': 32},
 {'zno_plane': (1, 1, 0), 'mgo_plane': (1, 1, 0), 'matches': 12},
 {'zno_plane': (1, 0, 1), 'mgo_plane': (1, 1, 0), 'matches': 67},
 {'zno_plane': (1, 0, 0), 'mgo_plane': (1, 1, 0), 'matches': 42},
 {'zno_plane': (0, 0, 1), 'mgo_plane': (1, 1, 0), 'matches': 294},
 {'zno_plane': (1, 1, 1), 'mgo_plane': (1, 0, 0), 'matches': 54},
 {'zno_plane': (1, 1, 0), 'mgo_plane': (1, 0, 0), 'matches': 44},
 {'zno_plane': (1, 0, 1), 'mgo_plane': (1, 0, 0), 'matches': 158},
 {'zno_plane': (1, 0, 0), 'mgo_plane': (1, 0, 0), 'matches': 102},
 {'zno_plane': (0, 0, 1), 'mgo_plane': (1, 0, 0), 'matches': 409}]

In [8]:
total_matches = 0

for item in cib_objects:
    total_matches += len(list(item["cib"].zsl_matches))

print("Total matches found:", total_matches)

Total matches found: 2289


In [9]:
all_interfaces = []

for item in cib_objects:
    cib = item["cib"]
    for term in cib.terminations:
        try:
            interfaces = list(
                cib.get_interfaces(
                    termination=term,
                    gap=1.50,
                    film_thickness=8,
                    substrate_thickness=4
                )
            )
            all_interfaces.extend(interfaces)
        except Exception:
            continue

print("Total interfaces generated:", len(all_interfaces))

Total interfaces generated: 4002


In [ ]:
print("CIB objects:", len(cib_objects))
print("Interfaces generated:", len(all_interfaces))

In [ ]:
interfaces_least_strain = min(all_interfaces,
           key=lambda x: x.interface_properties['von_mises_strain'])

print("Strain:", interfaces_least_strain.interface_properties['von_mises_strain']*100, "%")

In [ ]:
for interface in all_interfaces:

    print("ZnO transformation matrix:")
    print(interface.interface_properties["substrate_transformation"])

    print("MgO transformation matrix:")
    print(interface.interface_properties["film_transformation"])
    print("--------------------------")

In [ ]:
import numpy as np

for interface in all_interfaces:
    
    vecs = interface.interface_properties["substrate_sl_vectors"]
    
    a = vecs[0]
    b = vecs[1]
    
    area = np.linalg.norm(np.cross(a, b))
    
    interface.interface_properties["area"] = area

for interface in all_interfaces:
    print(
        interface.interface_properties["von_mises_strain"],
        interface.interface_properties["area"]
    )

In [ ]:
def score(interface):

    strain = interface.interface_properties["von_mises_strain"]
    area = interface.interface_properties["area"]

    # Normalized scoring

    return strain*10 + area/50

final_interface = min(all_interfaces, key=score)
final_interface

In [ ]:
def chemical_score(structure, film_indices):
    score = 0
    
    for i in film_indices:
        for j in range(len(structure)):
            if j in film_indices:
                continue
            
            d = structure.get_distance(i, j)
            
            species_i = structure[i].specie.symbol
            species_j = structure[j].specie.symbol
            
            if d < 2.5:
                # Favor O–Mg
                if (species_i == "O" and species_j == "Mg") or \
                   (species_i == "Mg" and species_j == "O"):
                    score -= 5
                
                elif species_i == species_j:
                    score += 10
    
    return score

base_score = interface_score(interface, film_indices)
final_score = base_score + chemical_score(interface, film_indices)

In [ ]:
def total_score_with_chem(interface, film_indices):
    
    # geometric score
    d_opt_struct, d_score = optimize_interface_distance(interface, film_indices)
    t_opt_struct, t_score = translate_film(d_opt_struct, film_indices)
    
    base_score = d_score + t_score
    
    # chemical score
    chem_score = chemical_score(t_opt_struct, film_indices)
    
    # strain contribution
    strain = 0
    if "strain" in interface.interface_properties:
        strain_tensor = interface.interface_properties["strain"]
        strain = np.linalg.norm(strain_tensor)
    
    # FINAL score
    total = base_score + chem_score + 10 * strain
    
    return total, t_opt_struct

best_interface = None
best_score = float("inf")

for interface in interfaces:
    
    film_indices = get_film_indices(interface)
    
    score_val, optimized_struct = total_score_with_chem(interface, film_indices)
    
    if score_val < best_score:
        best_score = score_val
        best_interface = optimized_struct

final_interface_chem = best_interface
final_interface_chem

In [ ]:
print("zno transformation matrix:")
print(final_interface.interface_properties["film_transformation"])

print("MgO transformation matrix:")
print(final_interface.interface_properties["substrate_transformation"])

In [ ]:
import numpy as np

f = final_interface.interface_properties["substrate_sl_vectors"][0]
s = final_interface.interface_properties["film_sl_vectors"][0]

angle = np.degrees(
        np.arccos(
        np.dot(f,s) / (np.linalg.norm(f)*np.linalg.norm(s))
    )
)

print("Rotation angle:", angle, 180 - angle)

In [ ]:
print("ZnO interface lattice vectors:")
print(final_interface.interface_properties["film_sl_vectors"])

print("MgO interface lattice vectors:")
print(final_interface.interface_properties["substrate_sl_vectors"])

In [ ]:
strain = final_interface.interface_properties["strain"]
print(strain)

In [ ]:
f = final_interface.interface_properties["film_sl_vectors"]
s = final_interface.interface_properties["substrate_sl_vectors"]

film_lengths = np.linalg.norm(f[:2], axis=1)
sub_lengths  = np.linalg.norm(s[:2], axis=1)

mismatch = abs(film_lengths - sub_lengths) / sub_lengths * 100
print(mismatch)

In [ ]:
print(final_interface.interface_properties)

In [ ]:
import numpy as np
from pymatgen.core.structure import Structure

def get_film_indices(interface):
    film_indices = []
    
    for i, site in enumerate(interface.sites):
        if site.specie.symbol in ["Zn", "O"]:
            film_indices.append(i)
            
    return film_indices

def interface_score(structure, film_indices, cutoff_min=1.2, cutoff_max=3.0):
    
    coords = structure.cart_coords
    score = 0
    
    for i in film_indices:
        for j in range(len(coords)):
            
            if j in film_indices:
                continue  # skip film-film
            
            d = np.linalg.norm(coords[i] - coords[j])
            
            if d < cutoff_min:
                score += 100 * (cutoff_min - d)
            
            elif d > cutoff_max:
                score += (d - cutoff_max)
    
    return score

In [ ]:
def optimize_interface_distance(interface, film_indices,
                                distances=np.linspace(1.5, 3.0, 12)):
    
    best_struct = None
    best_score = float("inf")
    
    for d in distances:
        trial = interface.copy()
        
        trial.translate_sites(
            film_indices,
            [0, 0, d],   # shift ONLY film
            frac_coords=False
        )
        
        s = interface_score(trial, film_indices)
        
        if s < best_score:
            best_score = s
            best_struct = trial
            
    return best_struct, best_score

In [ ]:
def translate_film(interface, film_indices, shifts=6):
    
    best_struct = None
    best_score = float("inf")
    
    for dx in np.linspace(0, 1, shifts):
        for dy in np.linspace(0, 1, shifts):
            
            trial = interface.copy()
            
            trial.translate_sites(
                film_indices,
                [dx, dy, 0],
                frac_coords=True
            )
            
            s = interface_score(trial, film_indices)
            
            if s < best_score:
                best_score = s
                best_struct = trial
                
    return best_struct, best_score

In [ ]:
def total_score(interface, base_score):
    
    strain = 0
    
    if "strain" in interface.interface_properties:
        strain_tensor = interface.interface_properties["strain"]
        strain = np.linalg.norm(strain_tensor)
    
    return base_score + 10 * strain

best_global = None
best_global_score = float("inf")

for interface in interfaces:  # your generated interfaces
    
    film_indices = get_film_indices(interface)
    
    # Step 1: fix interface distance
    d_opt_struct, d_score = optimize_interface_distance(interface, film_indices)
    
    # Step 2: fix in-plane registry
    t_opt_struct, t_score = translate_film(d_opt_struct, film_indices)
    
    base_score = d_score + t_score
    
    # Step 3: include strain if available
    final_score = total_score(interface, base_score)
    
    if final_score < best_global_score:
        best_global_score = final_score
        best_global = t_opt_struct

final_interface1 = best_global
final_interface1

In [ ]:
print(final_interface1.interface_properties)

In [ ]:
print(final_interface1.film)
print(final_interface1.substrate)

In [ ]:
import numpy as np

props = final_interface1.interface_properties

film_vectors = props["film_sl_vectors"]
sub_vectors = props["substrate_sl_vectors"]

film_normal = np.cross(film_vectors[0], film_vectors[1])
sub_normal = np.cross(sub_vectors[0], sub_vectors[1])

# normalize
film_normal = film_normal / np.linalg.norm(film_normal)
sub_normal = sub_normal / np.linalg.norm(sub_normal)

print("Film normal:", film_normal)
print("Substrate normal:", sub_normal)

In [ ]:
def approximate_miller(normal):
    # convert to smallest integer ratio
    normal = normal / np.min(np.abs(normal[np.nonzero(normal)]))
    return np.round(normal).astype(int)

print("Approx ZnO Miller:", approximate_miller(film_normal))
print("Approx MgO Miller:", approximate_miller(sub_normal))

In [ ]:
print(final_interface1.film.lattice)
print(final_interface1.substrate.lattice)

In [ ]:
#The optimized interface corresponds to ZnO (2 -1 2) aligned with MgO (210). 
#These are higher-index planes, indicating that the interface was selected based on 
#lattice matching and strain minimization rather than surface stability. 
#The computed von Mises strain (~0.039) suggests moderate mismatch. 
#Such high-index interfaces are less common experimentally but 
#can arise in computational optimization without Miller index constraints.